In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
pd.options.display.max_columns = 200
pd.set_option('future.no_silent_downcasting', True)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.api as sm

C:\Users\SilminaWota\AppData\Local\Temp\ipykernel_24144\2552498283.py:5: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


## Access CSV file

In [15]:
# Check Path and File
data_dir = Path.cwd() / "data"
print (data_dir)
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory does not exist: {data_dir}")
csv_files = list(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {data_dir}")
print("CSV files found:")
for f in csv_files:
    print(" -", f.name)

# Complete the data path
data_dir = Path.cwd() / "data" / "ChlorineSensor_TemperatureCalibrationCurve_Measures.csv"

c:\Users\SilminaWota\Documents\chlorinesensor\data
CSV files found:
 - ChlorineSensor_TemperatureCalibrationCurve_Measures.csv


In [16]:
# Load the data in a dateframe
df = pd.read_csv(data_dir)
df.head()

,sensor_num,temperature,signal_raw
0,1,10.2,145.3
1,1,12.5,152.8
2,1,15.1,161.5
3,1,17.8,170.2
4,1,20.0,178.9


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sensor_num   34 non-null     int64  
 1   temperature  34 non-null     float64
 2   signal_raw   34 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 948.0 bytes


## Signal analysis

In [18]:
df_electrode1 = df[df["sensor_num"] == 1]
df_electrode2 = df[df["sensor_num"] == 2]

In [19]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_electrode1["temperature"],
        y=df_electrode1["signal_raw"],
        mode="markers",
        name="Chlorine sensor 1",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_electrode2["temperature"],
        y=df_electrode2["signal_raw"],
        mode="markers",
        name="Chlorine sensor 2",
    )
)

fig.update_layout(
    title="Calibration curves: temperature vs signal",
    xaxis_title="Temperature (°C)",
    yaxis_title="Signal",
)

fig.show()

In [20]:
# Sensor 1
coef_1 = np.polyfit(df_electrode1["temperature"], df_electrode1["signal_raw"], deg=1)
fit_1 = np.poly1d(coef_1)
a1, b1 = coef_1   # sensor 1

# Sensor 2
coef_2 = np.polyfit(df_electrode2["temperature"], df_electrode2["signal_raw"], deg=1)
fit_2 = np.poly1d(coef_2)
a2, b2 = coef_2   # sensor 2

print(f"Sensor 1 slope: {a1:.3f}, intercept:{b1:.3f}")
print(f"Sensor 2 slope: {a2:.3f}, intercept:{b2:.3f}")

Sensor 1 slope: 3.472, intercept:109.253
Sensor 2 slope: 3.535, intercept:116.479


In [21]:
temp_range = np.linspace(
    df["temperature"].min(),
    df["temperature"].max(),
    100
)

fig = go.Figure()

# Raw points
fig.add_trace(go.Scatter(
    x=df_electrode1["temperature"], y=df_electrode1["signal_raw"],
    mode="markers", name="Chlorine sensor 1"
))
fig.add_trace(go.Scatter(
    x=df_electrode2["temperature"], y=df_electrode2["signal_raw"],
    mode="markers", name="Chlorine sensor 2"
))

# Fit lines
fig.add_trace(go.Scatter(
    x=temp_range, y=fit_1(temp_range),
    mode="lines", name="Calibration Sensor 1"
))
fig.add_trace(go.Scatter(
    x=temp_range, y=fit_2(temp_range),
    mode="lines", name="Calibration Sensor 2"
))

fig.update_layout(
    title="Calibration curves (linear fit)",
    xaxis_title="Temperature (°C)",
    yaxis_title="Signal",
)

fig.show()


In [22]:
# Statistical metrics
delta_slope = a2 - a1 # for sensitivity difference
#delta_intercept = b2 - b1 # for offset / bias
print(f"Sensitivity difference = {delta_slope:.3f}, difference in slope")
#print(f"Offset = {delta_intercept:.3f}")

def r_squared(x, y, model):
    y_pred = model(x)
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot

r2_1 = r_squared(df_electrode1["temperature"], df_electrode1["signal_raw"], fit_1)
r2_2 = r_squared(df_electrode2["temperature"], df_electrode2["signal_raw"], fit_2)

print(f"Sensor 1: R² = {r2_1:.3f}")
print(f"Sensor 2: R² = {r2_2:.3f}")

Sensitivity difference = 0.063, difference in slope
Sensor 1: R² = 1.000
Sensor 2: R² = 1.000


# Global calibration

In [23]:
x_all = df["temperature"]
y_all = df["signal_raw"]

coef_global = np.polyfit(x_all, y_all, deg=1)
fit_global = np.poly1d(coef_global)

a_g, b_g = coef_global

df["residual_global"] = df["signal_raw"] - fit_global(df["temperature"])
df.groupby("sensor_num")["residual_global"].agg(["mean", "std"])

df_coef = pd.DataFrame({
    "sensor": ["Sensor 1", "Sensor 2", "Global"],
    "slope": [a1, a2, a_g],
    "intercept": [b1, b2, b_g],
    "R2": [r2_1, r2_2, r_squared(x_all, y_all, fit_global)],
})

df_coef.head()

,sensor,slope,intercept,R2
0,Sensor 1,3.471651,109.253468,0.999889
1,Sensor 2,3.534757,116.479046,0.999825
2,Global,3.503763,112.849897,0.988631


In [24]:
# plot with confidence interval (CI) - uncertainty around the mean fitted line
# x/y as arrays
x = df["temperature"].to_numpy()
y = df["signal_raw"].to_numpy()

# Fit OLS: y = b + a*x
X = sm.add_constant(x)          # adds intercept column
model = sm.OLS(y, X).fit()

# Temperature grid for the line
x_grid = np.linspace(x.min(), x.max(), 200)
Xg = sm.add_constant(x_grid)

# Predictions + 95% CI
pred = model.get_prediction(Xg).summary_frame(alpha=0.05)
y_mean  = pred["mean"].to_numpy()
y_low   = pred["mean_ci_lower"].to_numpy()
y_high  = pred["mean_ci_upper"].to_numpy()

# ---- Plot ----
fig = go.Figure()

# Data points (colored by sensor_num)
for s in sorted(df["sensor_num"].dropna().unique()):
    sub = df[df["sensor_num"] == s]
    fig.add_trace(go.Scatter(
        x=sub["temperature"], y=sub["signal_raw"],
        mode="markers",
        name=f"Chlorine sensor {int(s)}"
    ))

# CI band (fill between low/high)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_grid, x_grid[::-1]]),
    y=np.concatenate([y_high, y_low[::-1]]),
    fill="toself",
    name="95% CI (mean)",
    line=dict(width=0),
    opacity=0.2,
))

# Global fit line
fig.add_trace(go.Scatter(
    x=x_grid, y=y_mean,
    mode="lines",
    name="Global calibration",
))

# Show coefficients in title (3 decimals)
b, a = model.params  # const, slope
fig.update_layout(
    title=f"Global calibration: signal = {a:.3f}·T + {b:.3f}",
    xaxis_title="Temperature (°C)",
    yaxis_title="Signal"
)

fig.show()
fig.write_image("plot_name.png", width=1200, height=600)

RuntimeError: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"


fig.write_image("plot_name.png", width=1200, height=600)

RuntimeError: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"
